In [3]:
"""
Cloud Watcher Agent — LangGraph Implementation
================================================
Architecture:
  [user_input] → [siva_node] → [agent_node] → [tools_node?] → [respond_node]

State Fields:
  - userInputNeeded : bool   — whether the graph is waiting for user input
  - AssistantMessage: str    — latest message to surface to the caller
  - observation     : dict   — normalized CloudWatch / EventBridge payload (Siva's output)
  - memory          : list   — full conversation history (agent memory)

Siva Node (primary / foundational):
  Normalises raw CloudWatch GetMetricStatistics responses *and* EventBridge
  event payloads into a unified state.observation dict before the agent runs.
  Maheshwar collaboration point: EventBridge rules wiring / payload shape lives
  in _normalize_eventbridge(); extend that section as the wiring solidifies.
"""

from __future__ import annotations

import json
import os
from datetime import datetime, timedelta
from typing import Any, Optional
from typing_extensions import TypedDict, Annotated
import operator

import boto3
import pytz
from dotenv import load_dotenv

from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

# ---------------------------------------------------------------------------
# AWS client
# ---------------------------------------------------------------------------
cloudwatch_client = boto3.client("cloudwatch")


# ---------------------------------------------------------------------------
# State
# ---------------------------------------------------------------------------
class CloudWatchState(TypedDict):
    """
    Central state object passed between every node.

    userInputNeeded  – set True when the graph needs the human to provide more
                       information before it can continue.
    AssistantMessage – the latest assistant-facing text the caller should
                       display / return to the user.
    observation      – normalised infrastructure observation produced by Siva.
    messages         – full conversation / tool-call history (agent memory).
    """

    userInputNeeded: bool
    AssistantMessage: str
    observation: dict
    # Annotated with operator.add so LangGraph merges list appends correctly
    messages: Annotated[list, operator.add]


# ---------------------------------------------------------------------------
# Tools
# ---------------------------------------------------------------------------
@tool
def get_cpu_metrics(instanceId: str, last_hours: int = 6, period: int = 1800) -> dict:
    """
    Fetches CPUUtilization metrics for an EC2 instance over the last N hours
    with configurable period intervals (in seconds).

    Args:
        instanceId: EC2 instance ID, e.g. i-0327bf109dc40d412
        last_hours:  How many hours back to query (default 6).
        period:      Granularity in seconds (default 1800 = 30 min).

    Returns:
        CloudWatch GetMetricStatistics response dict with IST-formatted timestamps.
    """
    ist = pytz.timezone("Asia/Kolkata")
    end_time = datetime.now(pytz.utc)
    start_time = end_time - timedelta(hours=last_hours)

    response = cloudwatch_client.get_metric_statistics(
        Namespace="AWS/EC2",
        MetricName="CPUUtilization",
        Dimensions=[{"Name": "InstanceId", "Value": instanceId}],
        StartTime=start_time,
        EndTime=end_time,
        Period=period,
        Statistics=["Average"],
        Unit="Percent",
    )

    for point in response["Datapoints"]:
        point["Timestamp"] = (
            point["Timestamp"].astimezone(ist).strftime("%Y-%m-%d %H:%M:%S IST")
        )

    response["Datapoints"].sort(key=lambda x: x["Timestamp"], reverse=True)
    return response


TOOLS = [get_cpu_metrics]
TOOLS_BY_NAME: dict[str, Any] = {t.name: t for t in TOOLS}


# ---------------------------------------------------------------------------
# LLM
# ---------------------------------------------------------------------------
# Replace model name with whatever is available in your environment.
# The original code used "gpt-5.4-nano"; adjust as needed.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(TOOLS)


# ---------------------------------------------------------------------------
# Helper: normalize a raw CloudWatch response into observation
# ---------------------------------------------------------------------------
def _normalize_cloudwatch(raw: dict) -> dict:
    """Convert a GetMetricStatistics response into a clean observation dict."""
    datapoints = raw.get("Datapoints", [])
    if not datapoints:
        return {"source": "cloudwatch", "status": "no_data", "datapoints": []}

    averages = [dp["Average"] for dp in datapoints]
    return {
        "source": "cloudwatch",
        "status": "ok",
        "metric": "CPUUtilization",
        "unit": "Percent",
        "datapoint_count": len(datapoints),
        "max_cpu": max(averages),
        "min_cpu": min(averages),
        "avg_cpu": sum(averages) / len(averages),
        "datapoints": datapoints,
    }


def _normalize_eventbridge(event: dict) -> dict:
    """
    Normalize an EventBridge event payload into the unified observation format.
    ─────────────────────────────────────────────────────────────────────────
    TODO (Maheshwar collab): Finalize EventBridge rule wiring and extend the
    field mapping below once the payload shape is confirmed.
    ─────────────────────────────────────────────────────────────────────────
    """
    return {
        "source": "eventbridge",
        "status": "ok",
        "event_source": event.get("source", "unknown"),
        "detail_type": event.get("detail-type", "unknown"),
        "detail": event.get("detail", {}),
        "time": event.get("time"),
        "region": event.get("region"),
        "account": event.get("account"),
    }


# ---------------------------------------------------------------------------
# Nodes
# ---------------------------------------------------------------------------

# ── Siva node (primary / foundational) ──────────────────────────────────────
def siva_node(state: CloudWatchState) -> dict:
    """
    Foundational observation-normalization node (Siva).

    Responsibilities:
      1. Inspect state.messages for any ToolMessage payloads that contain
         raw CloudWatch or EventBridge data.
      2. Normalise them into state.observation using the helpers above.
      3. Pass control downstream — does NOT modify userInputNeeded.

    This node runs first on every graph invocation so every downstream node
    (agent, tools, respond) always has a clean, typed observation available.
    """
    observation: dict = state.get("observation", {})
    messages = state.get("messages", [])

    for msg in reversed(messages):
        if not isinstance(msg, ToolMessage):
            continue
        try:
            payload = json.loads(msg.content) if isinstance(msg.content, str) else msg.content
        except (json.JSONDecodeError, TypeError):
            continue

        # Detect payload type and normalize
        if "Datapoints" in payload:  # CloudWatch
            observation = _normalize_cloudwatch(payload)
            break
        if "detail-type" in payload or "source" in payload:  # EventBridge
            observation = _normalize_eventbridge(payload)
            break

    return {"observation": observation}


# ── Agent node ───────────────────────────────────────────────────────────────
def agent_node(state: CloudWatchState) -> dict:
    """
    Calls the LLM with the full conversation history.
    Injects the current observation summary into the system prompt so the
    model always reasons over normalized data, not raw API responses.
    """
    system_content = (
        "You are Chandra, a cloud watcher agent that monitors cloud infrastructure.\n"
        "Analyse metrics, identify anomalies, and provide actionable observations.\n"
    )

    obs = state.get("observation", {})
    if obs:
        system_content += f"\nCurrent normalized observation:\n{json.dumps(obs, indent=2)}"

    system_msg = SystemMessage(content=system_content)
    history = state.get("messages", [])

    response: AIMessage = llm.invoke([system_msg, *history])
    return {"messages": [response]}


# ── Tools node ────────────────────────────────────────────────────────────────
def tools_node(state: CloudWatchState) -> dict:
    """Execute any tool calls requested by the agent."""
    last_message = state["messages"][-1]
    tool_results: list[ToolMessage] = []

    for tool_call in last_message.tool_calls:
        tool_fn = TOOLS_BY_NAME.get(tool_call["name"])
        if tool_fn is None:
            result = f"Tool '{tool_call['name']}' not found."
        else:
            result = tool_fn.invoke(tool_call["args"])

        tool_results.append(
            ToolMessage(
                content=json.dumps(result, default=str),
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": tool_results}


# ── Respond node ──────────────────────────────────────────────────────────────
def respond_node(state: CloudWatchState) -> dict:
    """
    Extracts the final assistant text, sets AssistantMessage, and marks
    userInputNeeded = False (the agent has finished its turn).
    """
    last_message = state["messages"][-1]
    assistant_text = (
        last_message.content
        if isinstance(last_message.content, str)
        else str(last_message.content)
    )
    return {
        "AssistantMessage": assistant_text,
        "userInputNeeded": False,
    }


# ── Clarification node ────────────────────────────────────────────────────────
def clarification_node(state: CloudWatchState) -> dict:
    """
    Called when the agent cannot proceed without more information from the user.
    Sets userInputNeeded = True and surfaces a clarifying question.
    """
    last_message = state["messages"][-1]
    question = (
        last_message.content
        if isinstance(last_message.content, str)
        else "Could you provide more details so I can assist you?"
    )
    return {
        "AssistantMessage": question,
        "userInputNeeded": True,
    }


# ---------------------------------------------------------------------------
# Routing
# ---------------------------------------------------------------------------
def route_after_agent(state: CloudWatchState) -> str:
    """
    After the agent responds:
      - If it made tool calls  → run tools
      - If it asked a question → route to clarification
      - Otherwise             → respond
    """
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    content = last_message.content or ""
    question_signals = ["?", "please provide", "could you", "what is", "which instance"]
    if any(sig in content.lower() for sig in question_signals):
        return "clarification"

    return "respond"


def route_after_tools(state: CloudWatchState) -> str:
    """After tools run, always go back through Siva → agent."""
    return "siva"


# ---------------------------------------------------------------------------
# Graph construction
# ---------------------------------------------------------------------------
def build_graph() -> StateGraph:
    builder = StateGraph(CloudWatchState)

    builder.add_node("siva", siva_node)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", tools_node)
    builder.add_node("respond", respond_node)
    builder.add_node("clarification", clarification_node)

    # Entry: every invocation starts at Siva
    builder.add_edge(START, "siva")
    builder.add_edge("siva", "agent")

    builder.add_conditional_edges(
        "agent",
        route_after_agent,
        {"tools": "tools", "respond": "respond", "clarification": "clarification"},
    )

    # After tools → re-enter Siva to normalize new payloads → agent
    builder.add_edge("tools", "siva")

    builder.add_edge("respond", END)
    builder.add_edge("clarification", END)

    return builder


# MemorySaver persists state across turns (agent memory per thread_id)
memory = MemorySaver()
graph = build_graph().compile(checkpointer=memory)


# ---------------------------------------------------------------------------
# Public runner helper
# ---------------------------------------------------------------------------
def run_agent(user_message: str, thread_id: str = "default") -> CloudWatchState:
    """
    Invoke the graph with a user message.

    Args:
        user_message: The natural-language query from the user.
        thread_id:    Conversation thread identifier for memory isolation.

    Returns:
        The final CloudWatchState with AssistantMessage and userInputNeeded set.
    """
    config = {"configurable": {"thread_id": thread_id}}
    initial_state: CloudWatchState = {
        "userInputNeeded": False,
        "AssistantMessage": "",
        "observation": {},
        "messages": [HumanMessage(content=user_message)],
    }
    final_state = graph.invoke(initial_state, config=config)
    return final_state


# ---------------------------------------------------------------------------
# CLI entrypoint
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    query = (
        "Give observations on CPU utilization for the last 8 hours "
        # "with 30-minute intervals in detail for instanceId i-0327bf109dc40d412"
    )

    print("=" * 60)
    print("Query:", query)
    print("=" * 60)

    result = run_agent(query, thread_id="session-001")

    print("\n[userInputNeeded]:", result["userInputNeeded"])
    print("\n[AssistantMessage]:\n", result["AssistantMessage"])

    if result.get("observation"):
        print("\n[Normalized Observation]:")
        print(json.dumps(result["observation"], indent=2, default=str))

Query: Give observations on CPU utilization for the last 8 hours 

[userInputNeeded]: True

[AssistantMessage]:
 Please provide the EC2 instance ID for which you would like to analyze the CPU utilization metrics over the last 8 hours.
